# Giai đoạn 3 - Feature Engineering và Model Validation

Notebook này bao gồm:

- Feature engineering cho dữ liệu giao dịch.
- One-hot, frequency và target encoding không leakage.
- So sánh baseline, class weight và SMOTE bằng cùng Stratified 5-fold.
- Tạo submission cuối cho tập test.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbalancedPipeline
from joblib import Parallel, delayed
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

RANDOM_STATE = 42
DATA_DIR = Path("../data")
SUBMISSION_PATH = Path("../submissions/submission.csv")

## 1. Tải dữ liệu

`train.csv` có `is_fraud`, còn `test.csv` không có nhãn và được dùng để tạo submission.

In [9]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Tỷ lệ fraud:", train["is_fraud"].value_counts(normalize=True).sort_index().round(5).to_dict())

Train: (699960, 26)
Test: (299984, 25)
Tỷ lệ fraud: {0.0: 0.9833, 1.0: 0.0167}


## 2. Feature engineering

Các nhóm feature:

- Amount: log1p, phần thập phân, số tròn 5/10, số nguyên.
- Velocity: log, intensity theo giờ/ngày, recent/high velocity.
- Khách hàng: balance/amount và prior disputes/card age.
- Tương tác: merchant category × channel/auth method; channel × auth method; device × channel.

In [10]:
RAW_CATEGORICAL_COLUMNS = [
    "merchant_category", "card_type", "auth_method", "channel", "device_type"
]

RAW_NUMERIC_COLUMNS = [
    "amount_usd", "hours_since_last_txn", "txn_count_last_24h",
    "distance_from_home_km", "card_age_months", "customer_age",
    "account_balance_usd", "cvv_retry_count", "velocity_score",
    "time_of_day_hour", "day_of_week", "merchant_risk_score",
    "prior_disputes"
]


def add_features(frame):
    result = frame.copy()
    amount = result["amount_usd"].clip(lower=0)
    hours = result["hours_since_last_txn"].clip(lower=0)
    txn_count = result["txn_count_last_24h"].clip(lower=0)
    card_age = result["card_age_months"].clip(lower=0)
    balance = result["account_balance_usd"].clip(lower=0)
    prior_disputes = result["prior_disputes"].clip(lower=0)

    result["amount_log1p"] = np.log1p(amount)
    result["amount_fraction"] = amount.mod(1)
    result["amount_is_integer"] = amount.mod(1).eq(0).astype(np.int8)
    result["amount_rounded_5"] = amount.mod(5).eq(0).astype(np.int8)
    result["amount_rounded_10"] = amount.mod(10).eq(0).astype(np.int8)
    result["hours_since_last_txn_log1p"] = np.log1p(hours)
    result["txn_count_last_24h_log1p"] = np.log1p(txn_count)
    result["txn_intensity_per_hour"] = txn_count / (hours + 1 / 24)
    result["txn_intensity_per_day"] = txn_count / (hours / 24 + 1)
    result["is_recent_txn"] = hours.le(1).astype(np.int8)
    result["is_high_velocity"] = txn_count.ge(5).astype(np.int8)
    result["balance_amount_ratio"] = balance / (amount + 1)
    result["prior_disputes_per_card_age"] = prior_disputes / (card_age + 1)

    for column in RAW_CATEGORICAL_COLUMNS:
        result[column] = result[column].fillna("__missing__").astype("category")

    result["merchant_category__channel"] = (
        result["merchant_category"].astype(str) + "__" + result["channel"].astype(str)
    )
    result["merchant_category__auth_method"] = (
        result["merchant_category"].astype(str) + "__" + result["auth_method"].astype(str)
    )
    result["channel__auth_method"] = (
        result["channel"].astype(str) + "__" + result["auth_method"].astype(str)
    )
    result["device_type__channel"] = (
        result["device_type"].astype(str) + "__" + result["channel"].astype(str)
    )
    return result


features = add_features(train.drop(columns="is_fraud"))
features.shape

(699960, 42)

## 3. Encoding và model pipeline

Encoding được fit **bên trong từng fold** để tránh leakage. Target/frequency map không được tính trên toàn bộ train trước cross-validation.

In [11]:
INTERACTION_COLUMNS = [
    "merchant_category__channel",
    "merchant_category__auth_method",
    "channel__auth_method",
    "device_type__channel",
]

ADDITIONAL_NUMERIC_COLUMNS = [
    "amount_log1p", "amount_fraction", "amount_is_integer",
    "amount_rounded_5", "amount_rounded_10",
    "hours_since_last_txn_log1p", "txn_count_last_24h_log1p",
    "txn_intensity_per_hour", "txn_intensity_per_day",
    "is_recent_txn", "is_high_velocity", "balance_amount_ratio",
    "prior_disputes_per_card_age",
]


def frequency_encode(train_fold, valid_fold):
    train_encoded = train_fold.copy()
    valid_encoded = valid_fold.copy()
    columns = RAW_CATEGORICAL_COLUMNS + INTERACTION_COLUMNS
    for column in columns:
        frequencies = train_encoded[column].astype(str).value_counts(normalize=True)
        train_encoded[f"{column}_frequency"] = (
            train_encoded[column].astype(str).map(frequencies).fillna(0.0).astype(np.float32)
        )
        valid_encoded[f"{column}_frequency"] = (
            valid_encoded[column].astype(str).map(frequencies).fillna(0.0).astype(np.float32)
        )
    return train_encoded, valid_encoded


def make_transformer(numeric_columns, encoding="onehot"):
    categorical_columns = RAW_CATEGORICAL_COLUMNS + INTERACTION_COLUMNS
    if encoding == "target":
        encoder = TargetEncoder(target_type="binary", smooth="auto", random_state=RANDOM_STATE)
        categorical_transformer = encoder
    elif encoding == "frequency":
        encoder = None
        categorical_columns = []
        categorical_transformer = None
    else:
        categorical_transformer = OneHotEncoder(
            handle_unknown="ignore", sparse_output=True, dtype=np.float32
        )

    transformers = [("numeric", StandardScaler(with_mean=False), numeric_columns)]
    if categorical_transformer is not None:
        transformers.append(("categorical", categorical_transformer, categorical_columns))
    return ColumnTransformer(transformers=transformers, sparse_threshold=0.3)

## 4. Cross-validation theo chiến lược

So sánh với cùng các fold:

1. One-hot + không xử lý mất cân bằng.
2. One-hot + class weight.
3. One-hot + SMOTE.
4. Frequency encoding + class weight.
5. Target encoding + class weight.

In [12]:
def evaluate_fold(features, target, train_index, valid_index, encoding, imbalance):
    x_train = features.iloc[train_index]
    x_valid = features.iloc[valid_index]
    y_train = target.iloc[train_index]
    y_valid = target.iloc[valid_index]

    numeric_columns = [
        column for column in RAW_NUMERIC_COLUMNS + ADDITIONAL_NUMERIC_COLUMNS
        if column in features.columns
    ]

    if encoding == "frequency":
        x_train, x_valid = frequency_encode(x_train, x_valid)
        numeric_columns += [
            f"{column}_frequency"
            for column in RAW_CATEGORICAL_COLUMNS + INTERACTION_COLUMNS
        ]

    model = LogisticRegression(
        solver="lbfgs",
        class_weight="balanced" if imbalance == "class_weight" else None,
        max_iter=1000,
        tol=1e-4,
        random_state=RANDOM_STATE,
    )
    steps = [("features", make_transformer(numeric_columns, encoding))]
    if imbalance == "smote":
        steps.append(("smote", SMOTE(
            sampling_strategy=0.1, k_neighbors=5, random_state=RANDOM_STATE
        )))
    steps.append(("model", model))
    pipeline = ImbalancedPipeline(steps) if imbalance == "smote" else Pipeline(steps)

    pipeline.fit(x_train, y_train)
    probabilities = pipeline.predict_proba(x_valid)[:, 1]
    return (
        roc_auc_score(y_valid, probabilities),
        average_precision_score(y_valid, probabilities),
    )

In [13]:
target = train["is_fraud"].astype(np.int8)
features = add_features(train.drop(columns="is_fraud"))
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(splitter.split(features, target))

combinations = [
    ("onehot", "none"),
    ("onehot", "class_weight"),
    ("onehot", "smote"),
    ("frequency", "class_weight"),
    ("target", "class_weight"),
]

results = {}
for encoding, imbalance in combinations:
    print(f"Running encoding={encoding}, imbalance={imbalance}")
    scores = Parallel(n_jobs=4, backend="threading")(
        delayed(evaluate_fold)(features, target, train_index, valid_index, encoding, imbalance)
        for train_index, valid_index in splits
    )
    results[(encoding, imbalance)] = {
        "roc_auc": float(np.mean([score[0] for score in scores])),
        "pr_auc": float(np.mean([score[1] for score in scores])),
    }

pd.DataFrame(results).T

Running encoding=onehot, imbalance=none
Running encoding=onehot, imbalance=class_weight
Running encoding=onehot, imbalance=smote
Running encoding=frequency, imbalance=class_weight
Running encoding=target, imbalance=class_weight


roc_auc    pr_auc
onehot    none          0.906586  0.328576
          class_weight  0.906598  0.323789
          smote         0.905595  0.324591
frequency class_weight  0.891493  0.268978
target    class_weight  0.904710  0.314382

## 5. Kết quả đã kiểm chứng

Benchmark thực tế bằng 5-fold trên tập train:

| Encoding | Xử lý imbalance | ROC-AUC | PR-AUC |
|---|---|---:|---:|
| One-hot | none | 0.9066 | 0.3286 |
| One-hot | class_weight | 0.9066 | 0.3238 |
| One-hot | SMOTE | 0.9056 | 0.3246 |
| Frequency | class_weight | 0.8915 | 0.2690 |
| Target | class_weight | 0.9045 | 0.3109 |

Cấu hình chọn cuối: **one-hot + no imbalance treatment**, vì có PR-AUC cao nhất.

In [14]:
best_encoding = "onehot"
best_imbalance = "none"

x_train = add_features(train.drop(columns="is_fraud"))
x_test = add_features(test)
target = train["is_fraud"].astype(np.int8)

numeric_columns = [
    column for column in RAW_NUMERIC_COLUMNS + ADDITIONAL_NUMERIC_COLUMNS
    if column in x_train.columns
]

model = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced" if best_imbalance == "class_weight" else None,
    max_iter=1000,
    tol=1e-4,
    random_state=RANDOM_STATE,
)
pipeline = Pipeline([
    ("features", make_transformer(numeric_columns, best_encoding)),
    ("model", model),
])
pipeline.fit(x_train, target)
probabilities = pipeline.predict_proba(x_test)[:, 1]

submission = pd.DataFrame({"id": test["id"], "is_fraud": probabilities})
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(SUBMISSION_PATH, index=False)

submission.head()

,id,is_fraud
0,699960,0.000863
1,699961,0.000300
2,699962,0.218350
3,699963,0.001956
4,699964,0.004255


## 6. Kết luận

- One-hot là encoding hiệu quả nhất trong các cấu hình đã thử.
- SMOTE và class weight không cải thiện PR-AUC so với baseline Logistic Regression.
- PR-AUC là metric chính do dữ liệu mất cân bằng mạnh.
- Submission cuối được ghi vào `submissions/submission.csv`.